[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_05.ipynb)

**Before editing:** choose **File → Save a copy in Drive**.

[Course website](https://loem-ms.github.io/programming-for-information-systems-site/) · [Setup and saving](https://loem-ms.github.io/programming-for-information-systems-site/content/setup/)

**Ready to run:** this notebook contains its support files. Connect to a normal Python runtime and run the setup cell below. No ZIP, upload, support-file download, installation or API key is needed. Opening this link alone does not save a personal copy. Rerun setup after a runtime restart. Existing support files are preserved. Read and run one cell at a time; complete marked student tasks yourself. Save your notebook and download any edited or created files separately before the temporary runtime ends.


<a id="start"></a>
# Week 05 · Text input, conversion and exceptions

[Start here](#start) · [Session 1](#session-1) · [Session 2](#session-2) · [Project studio](#project-studio) · [Finish](#finish)

**This notebook is your working home for the whole week.** Read the explanations, run the examples, write the practice here, then continue to the project. [Python Basics](https://loem-ms.github.io/programming-for-information-systems-site/content/learning-materials/python-basics/) is an optional syntax lookup; you do not need to leave this notebook to complete the core work.

**Required route:** read → predict → run → modify → check → next. Complete the 3 short core practice tasks and the marked project work. Optional follow-ups are clearly labelled and can wait.

**Save:** keep your edited `Week_05.ipynb`, any files you create, and a separate backup. Send assessed work to [submit.progmis@gmail.com](mailto:submit.progmis@gmail.com); exact deadlines and file names remain separately announced. The exact deadline/file-naming rule will be announced; shared Issues are for small questions, not complete submissions.

**Run order:** start with a fresh kernel and run downward. Teaching examples are complete; cells tagged or labelled as your work intentionally remain unfinished. Running the untouched starter is not completion.

### Prepare your workspace

Run the supplied setup cell after every runtime restart. It restores embedded programs, data and diagrams without downloading or uploading anything. Existing files are never overwritten. This helper is not a programming exercise. Read the later task instructions before running unfinished student work.


In [ ]:
# @title Prepare the course files (run after each runtime restart)
# Supplied programs, starter policy, CSV fixtures and diagrams are embedded here.
# This cell uses only Python's standard library. It makes no network request.
import base64 as _course_base64
import hashlib as _course_hashlib
import json as _course_json
import os as _course_os
from pathlib import Path as _CoursePath
import zlib as _course_zlib

_course_payload = '...'
_course_raw = _course_zlib.decompress(_course_base64.b64decode(_course_payload))
if _course_hashlib.sha256(_course_raw).hexdigest() != 'fe52e3238eed1aa62fa6723f190eadc35faa8184444c13d1ed134ad3bb2c9114':
    raise ValueError("Embedded course files are damaged; get a fresh notebook.")
_course_files = _course_json.loads(_course_raw)
_course_key = 'week_05_fe52e3238eed'
_course_roots = globals().setdefault("_course_workspace_roots", {})
if _course_key in _course_roots:
    _course_root = _CoursePath(_course_roots[_course_key])
elif all((_CoursePath.cwd() / _name).is_file() for _name in _course_files):
    # A complete extracted offline pack can be used in place.
    _course_root = _CoursePath.cwd()
else:
    _course_root = _CoursePath.cwd() / ".course_files" / _course_key
if _course_root.is_symlink() or any(_p.is_symlink() for _p in _course_root.parents):
    raise ValueError("Refusing a symbolic link in the course workspace")
_course_root.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _course_files.items():
    _relative = _CoursePath(_name)
    if _relative.is_absolute() or ".." in _relative.parts:
        raise ValueError("Unsafe embedded support path")
    _target = _course_root / _relative
    if _target.is_symlink() or any(_p.is_symlink() for _p in _target.parents):
        raise ValueError("Refusing a symbolic link in the course workspace")
    _target.parent.mkdir(parents=True, exist_ok=True)
    if _target.exists() and not _target.is_file():
        raise ValueError("Course support path is not a regular file: " + str(_target))
    if not _target.exists():
        with _target.open("xb") as _output:
            _output.write(_course_base64.b64decode(_encoded))
_course_workspace_roots[_course_key] = str(_course_root.resolve())
_course_os.chdir(_course_root)
print("Course files ready:", _CoursePath.cwd())
print("Existing files were kept. Rerunning setup never replaces your edits.")
print("Save your notebook and download created/edited files before the runtime ends.")


## This week: required, supplied and optional

Start with the coordination need, connect it to the new Python words and syntax, trace the worked example, make the guided change, predict the independent variation, and apply it to the Session 3 component.

- **Required evidence:** Complete the importer behavior and justify each supplied record’s outcome.
- **Already supplied — use it; do not rebuild it:** The CSV fixture, command-line scaffold and audit pattern are supplied.
- **Optional extension:** Additional malformed rows are optional after the required fixture passes.

**Stop point:** once the required evidence is saved and you can explain it, stop. You may return to optional material later.


## Session 1 · Python foundations
<a id="session-1"></a>

Start with the general Python concept and the self-contained example below. Predict, run and explain it before writing your own code in Session 2. A short optional Food Rescue note shows one application; no project knowledge is needed for this lesson.

### External text, useful errors and an audit trail

Programs often receive text from a form, CSV file or terminal command. A field containing `"4"` is a string, so multiplying it repeats characters. Convert it before doing arithmetic. `int(raw)` returns an integer when the text represents one, or raises `ValueError` when it cannot convert. `"4.5"` is not integer text. Start by stating what the input is allowed to mean.

**Conversion and validation are separate.** Conversion of `"0"` succeeds, but a positive-count contract rejects zero. First convert, then check the permitted range, then return the value. `raise ValueError("count must be positive")` makes a deliberate refusal explicit. A successful conversion tells you the type; a range check tells you whether that value satisfies the stated contract.

**A failure changes control flow.** Inside `try`, statements run until one raises an exception. Python skips the remaining statements and looks for a matching `except`. Put the smallest relevant operation in the try block and catch the expected `ValueError`. Catching every error would hide programming mistakes such as a misspelled variable. A handler can record the raw text and a reason, then continue with the next record in a loop. A function that raises did not return a usable count.

**Check fields before using them.** `row.get("count_text")` returns the value if present, otherwise `None`. Check for a missing field before calling string methods. `raw.strip()` removes outside whitespace and returns a new string. It does not modify the original. Retain the original row or text in the audit result so a reader can see what was received. For this week's exercises, supplied present fields are strings; general type checking comes in Week 8.

**Compose small helpers.** One function converts and checks one count. Another can use that helper to return an accepted or rejected result with source ID, original text and reason. A loop builds an audit list without dropping rejected rows. Use tests for valid text, whitespace, zero, a negative count and nonnumeric text. Retaining each outcome makes debugging and later explanation possible. A function that raises must not silently substitute a believable number.

**Optional application example.** Food Rescue applies this pattern to proposed donor data. Its supplied duplicate and human-review rules belong to that application; they do not change how int, raise or try/except work. Learn the small text example first, then apply the method to the project if useful.

### Learn one operation at a time

Read and run these small examples in order before the combined example. Predict each output, point to the operation that produced it, and pause for clarification if it is unclear. They explain the existing lesson; they are not an extra homework list or extra assessed deliverables. The practice cells later remain yours to write.


### Small example 1 · Clean and convert one string

strip returns text with surrounding whitespace removed; int converts that text to an integer. The original string remains unchanged. repr shows quotes and spaces so you can see the input. Predict a quoted spaced value, 3, and int.


In [ ]:
raw = " 3 "
cleaned = raw.strip()
count = int(cleaned)
print(repr(raw))
print(count, type(count).__name__)


### Small example 2 · Handle a conversion failure

int cannot convert this word, so the rest of try is skipped and except ValueError runs. Predict Not an integer, then Continue. The handler does not retry the conversion or produce a valid count.


In [ ]:
raw = "many"
try:
    count = int(raw)
    print("Converted:", count)
except ValueError:
    print("Not an integer")
print("Continue")


### Small example 3 · Reject a value after successful conversion

Zero converts successfully but is outside this positive-count contract. raise makes that refusal explicit. Predict count must be positive. Conversion and permission to use the value are separate checks.


In [ ]:
raw = "0"
try:
    count = int(raw)
    if count <= 0:
        raise ValueError("count must be positive")
    print("Accepted:", count)
except ValueError as error:
    print(error)


### Small example 4 · Check a missing field before string methods

get returns None when a key is absent. Check that before calling strip; None is not text. Predict Missing count. A missing field is not the same as a zero value.


In [ ]:
row = {"source_id": "A"}
raw = row.get("count_text")
if raw is None:
    print("Missing count")
else:
    print(raw.strip())


### Small example 5 · Copy a row before normalizing it

dict.copy makes a separate outer dictionary. These fields contain only strings and integers, so replacing a field in the copy leaves the raw source unchanged. Predict str for the original and int for the normalized copy. This is the row-copy operation required by the validator task; nested data would need a different copying decision.


In [ ]:
raw_row = {"count": "3"}
normalized = raw_row.copy()
normalized["count"] = int(raw_row["count"])
print(type(raw_row["count"]).__name__)
print(type(normalized["count"]).__name__)
assert raw_row["count"] == "3"
assert normalized["count"] == 3


### Worked example: predict, then trace

Read the complete example once. Write each expected result before running. Point to the statement responsible for each value.

In [ ]:
raw_values = [" 2 ", "many"]
for raw in raw_values:
    try:
        count = int(raw.strip())
    except ValueError:
        print("Cannot convert:", repr(raw))
    else:
        print("Converted:", count)


The first input becomes integer 2. The second enters the handler. The else block runs only when try completed without an exception. This is a conversion demonstration; the practice adds the positive-range contract and preserved evidence.

### Research 3 exchange · during Session 1

Share your prepared Research 3 response on **input validation**: one invalid or conflicting input, your proposed check and who must resolve the ambiguity. Bring your source or supplied reading-card reference, example and reasoning. Record one question or useful piece of feedback below; retain your individual response. This is the scheduled exchange, not a new research assignment.

**My feedback and next step:** …

### Ready to write

Explain the input, the result and one important Python step without reading the output. Continue to [Session 2](#session-2). Apply the same general skill to each problem after reading its own stated rule.

## Session 2 · Hands-on practice
<a id="session-2"></a>
<a id="practice-week-05"></a>

**Five leveled coding problems:** P1 and P2 are easy problems with approach hints, not ready-made code. P3 is the required core implementation. P4 applies the skill and P5 is a harder challenge. Complete P1–P3 first; P4–P5 are optional.

Write a prediction, then write the program yourself. Keep the first result, explain a mismatch and retest. You may use the Session 1 syntax explanation. Beginners can work with a partner and request a hint; faster learners should attempt P4–P5 instead of waiting. Each learner saves their own code and explanation.

### Practice route

- **P1–P2 · Easy:** Use the approach hints. Write the code yourself in a blank cell.
- **P3 · Core build:** Implement a complete small program. Write expected results and assertions.
- **P4–P5 · Optional:** Apply the skill to a new case, then attempt a harder challenge if time permits.

### P1 · Easy: convert and label a value

**Required core · Level 1.**

From a blank cell, store raw_count as " 3 ". Strip the whitespace, convert to count, calculate planned using 2 portions per container and print both the integer count and 6 planned portions. Hint: strip returns text; int returns an integer. Assert count equals 3 and planned equals 6. Keep raw_count unchanged and print it with repr.

In [ ]:
# Write your code here. Predict before running.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P2 · Easy: write a positive-count parser

**Required core · Level 1.**

Write parse_positive(raw) from a blank cell. Convert the string, reject values <=0 with ValueError, and return an allowed integer. Hint: conversion, range check, return. Assert "4" returns 4. Write try/except checks showing "0", "-1" and "many" are refused; record which fail conversion and which fail the range check. Do not return zero on failure.

In [ ]:
# Write your code here. Predict before running.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P3 · Core: preserve each outcome

**Required core · Level 2.**

Use raw counts ["3", "0", "many", " 2 "]. Write your own loop calling parse_positive inside try/except. Append one audit dictionary per input with raw and status; add count for "proposed", and reason for "rejected". Expected statuses: proposed, rejected, rejected, proposed; counts 3 and 2. Assert four audit rows remain in input order and original text is preserved. These results propose data only, never set reviewed=True.

In [ ]:
# Write your code here. Predict the result before you run it.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P4 · Apply: distinguish missing from invalid

**Optional extension · Level 3.**

Write inspect_row(row) for small practice records containing source_id and optional count_text. Return source_id, status and reason. Missing or blank count_text is "held" with a clarification reason; valid positive text is "proposed" with count; nonnumeric or nonpositive text is "rejected". Reuse parse_positive. Check missing, blank, "2", "0" and "many". Do not drop source_id or invent a count.

In [ ]:
# Write your code here. Predict the result before you run it.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P5 · Challenge: collect a complete audit summary

**Optional extension · Level 4.**

Write summarize_audit(audit) that returns counts for proposed, held and rejected, plus total proposed containers. Use a loop and named accumulators. Test one proposed row with 3 containers, one held and one rejected: 1,1,1 and 3. Test an empty audit. Assert the status counts sum to the number of source rows; explain why this catches silently dropped input.

In [ ]:
# Write your code here. Predict the result before you run it.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

P4–P5 add missing-field and reconciliation work after the core. The existing project importer and duplicate policy remain supplied; neither optional task authorizes or reserves food.

### Save and continue

Keep your own code, independent expectations and observed evidence. Restart and rerun the completed practice cells to catch hidden state. Leave an unfinished optional problem clearly marked. Continue to the [scheduled project or assessment](#project-studio).

## Session 3 · Project studio
<a id="project-studio"></a>

**Read:** the project context below. **Predict:** write an expected result before implementing. **Run/modify:** complete the marked task using the supplied starting data. **Check:** compare evidence with a peer and retain a revision. **Next:** [Finish and save](#finish).

Each week supplies its own data; an unfinished earlier notebook is not a prerequisite.

### The people’s task this week

**Someone needs help:** Messages and table rows may contain missing or unclear counts.

**Your small part:** Check each incoming row and keep a reason for its outcome.

![Week 5: the person’s need, your task, and the useful result.](https://loem-ms.github.io/programming-for-information-systems-site/learning_materials/assets/diagrams/course_week_05.svg)

**Picture one example:** Missing count → ask; do not guess.

**A useful result:** A human can follow up unclear information before it becomes an available offer.

Use the supplied starting files and the task instructions below. The diagram explains the purpose; it adds no separate assignment.

In [ ]:
# @title Show this lesson’s embedded diagram(s)
try:
    from IPython.display import SVG, display
except ImportError:
    print("Diagrams are in assets/diagrams; Jupyter or Colab displays them below.")
else:
    display(SVG(filename='assets/diagrams/course_week_05.svg'))


### Project context and working rules

Campus Food Rescue is a fictional Cambodian university volunteer team coordinating offers from nearby food businesses with community partner coordinators. All records and approvals are invented. This course teaches coordination software, not food-safety certification. A human coordinator supplies the review decision; Python must not infer suitability from a numeric field.

Start with a fresh Python 3 kernel. Read and run the supplied examples in order, then complete each marked task. Restart and run the whole notebook when checking your completed work; running the untouched starter does not complete the exercises. Examples use the standard library. Counts are containers and declared planning portions, never money or verified people served. A reservation is not a collection or delivery. Student lab cells are deliberately incomplete and are labelled below.

### A reusable calculation now receives external text

Campus Food Rescue can calculate a plan and return a review-first workload label. The coordinator now receives donor CSV rows, while volunteers want command-line inputs rather than source edits. Both paths bring text into the program.

A valid-looking number is not an approved offer. This week's importer produces **validated proposals**, clarification holds and rejections. It does not increase `available_containers`, authorize a reservation or record delivery. The donor clarifies contents; the human coordinator supplies review and arrangements separately.

![Validate donor claims, preserve evidence, and classify each row.](https://loem-ms.github.io/programming-for-information-systems-site/learning_materials/assets/diagrams/w05_import.svg)

Practise CSV reading, conversion and range validation; use the supplied duplicate policy to reconcile rows. Optional lookup: [exceptions and CSV](https://loem-ms.github.io/programming-for-information-systems-site/content/learning-materials/python-basics/#python-basics-b09) and [CLI inputs](https://loem-ms.github.io/programming-for-information-systems-site/content/learning-materials/python-basics/#python-basics-b16).

In [ ]:
# @title Show this lesson’s embedded diagram(s)
try:
    from IPython.display import SVG, display
except ImportError:
    print("Diagrams are in assets/diagrams; Jupyter or Colab displays them below.")
else:
    display(SVG(filename='assets/diagrams/w05_import.svg'))


### Project references · CSV input and optional CLI

The following supplied examples apply the earlier Python concepts. CSV supports today’s row-validation task; command arguments are optional.

### Supplied CSV context: CSV fields arrive as strings

Use this supplied example to connect the Python conversion lesson with the project input. No additional practice task is added.

In [ ]:
import csv
from io import StringIO
text = """batch_id,reported_containers
DEMO,4
"""
row = next(csv.DictReader(StringIO(text)))
print(row["reported_containers"])
print(type(row["reported_containers"]).__name__)
assert row["reported_containers"] == "4"


### Optional supplied example: argparse: read a typed argument

Use this supplied example to connect the Python conversion lesson with the project input. No additional practice task is added.

In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument("--containers", type=int, default=4)
args = parser.parse_args(["--containers", "2"])
print(args.containers)
assert args.containers == 2


### One small implementation task: validate a row

Use the six-row file below. The supplied runner handles grouping, duplicate holds and audit assembly. Your core task is to implement **only `validate_offer_row(row)`**:

1. Strip and require `batch_id`.
2. Convert `portions_per_container` and `reported_containers` to integers.
3. Reject nonpositive values with `ValueError` and a useful reason.
4. Return a copy of the row containing the normalized ID and the two integers.

**Two core checks:** one valid row must return the expected integers; a row with `reported_containers = "many"` must raise `ValueError`. Then run the small six-row task and explain one proposed, held and rejected record. The runner supplies the duplicate policy: every valid row sharing an ID is held for clarification. Do not infer which conflicting declaration is authoritative from row order.

The required result is a row-level audit, not an availability update. Writing your own grouping algorithm, exploring alternative duplicate policies, reversing rows and extra invalid cases are **optional extensions** after the core task.

In [ ]:
def validate_offer_row(row):
    """Return normalized values; raise ValueError for invalid identity/counts."""
    raise NotImplementedError("Implement the four row-validation steps above")


### Supplied runner · use it without rebuilding the grouping algorithm

The runner preserves each source record and calls your validator. It groups only valid candidates, holds repeated IDs, and returns proposals and one audit disposition per source row. A record ordinal identifies a CSV record, not a physical file line.

**Run this supplied cell now** to load the fixture and helper; it checks the six-row fixture without running your unfinished validator. After completing your validator, pass it as the second argument: `build_import_audit(offer_file, validate_offer_row)`.

In [ ]:
import csv
from io import StringIO
from collections import Counter

offer_file = 'batch_id,description,portions_per_container,reported_containers\nQ11,Bread packs,3,4\nQ12,Rice meal containers,5,2\nQ11,Bread packs,4,4\nQ13,Vegetable trays,2,3\n,Unidentified trays,3,2\nQ14,Fruit containers,2,many\n'

def build_import_audit(text, validate_row):
    inspected = []
    for ordinal, raw in enumerate(csv.DictReader(StringIO(text)), start=2):
        try:
            candidate = validate_row(raw)
        except ValueError as error:
            inspected.append((ordinal, raw, None, str(error)))
        else:
            inspected.append((ordinal, raw, candidate, None))
    counts = Counter(item[2]["batch_id"] for item in inspected
                     if item[2] is not None)
    proposals, audit = [], []
    for ordinal, raw, candidate, error in inspected:
        if error is not None:
            status, reason = "rejected", error
            action = "Request a corrected value from the donor"
        elif counts[candidate["batch_id"]] > 1:
            status = "held"
            reason = "Repeated batch ID; clarify the original claims"
            action = "Coordinator asks the donor to resolve the claims"
        else:
            status, reason = "proposed", "Valid unique claim; human review still needed"
            action = "Coordinator reviews suitability and arrangements"
            proposals.append(candidate)
        audit.append({"record_ordinal": ordinal, "raw": raw,
                      "status": status, "reason": reason, "next_action": action})
    return proposals, audit

# The supplied fixture must contain six source records.
assert len(list(csv.DictReader(StringIO(offer_file)))) == 6


### Predict → check → explain

Before calling the supplied runner, write the expected disposition for every source record. After you implement the validator, run the two core checks and the six-row audit in the cell below. Explain why the audit has six entries and why two conflicting Q11 records remain visible.

Keep the raw fields, reason and next action. A `proposed` row is not reviewed availability. For optional extension, test a negative count or reverse the source order and explain whether the proposal set changes.

In [ ]:
# Core check 1: a valid row returns integer counts and the expected ID.
# Core check 2: reported_containers = "many" raises ValueError.
# Then run:
# proposals, audit = build_import_audit(offer_file, validate_offer_row)
# Record all six predicted/observed dispositions and explain one of each status.


### Peer audit and individual explanation

Give a partner the audit and original rows. Explain one proposed row, one held row and one rejected row with its next action. Record a finding and your response. If a donor says “four boxes like last time,” name what still needs clarification.

In [ ]:
# Your audit finding, explanation and revision go here.


<a id="research"></a>
### Research 4 · how programs keep data

**Research how programs keep data.** Find out how files or databases let a program retrieve information after it closes.

**Possible directions:** Look at record IDs, saved fields and retrieval. Consider a fact whose original value should be kept even after a later correction.

**A short summary example:** “Plan EX-8 saved 2 containers at 4 portions each. If the current unit becomes 5, keeping the original value still explains the old total of 8.”

Briefly explain what you found in your own words and name the sources you used. You may use a short note or slides; the example above is guidance, not a required format. Prepare after class and bring one finding to Week 6, Session 1.

The course Research guide explains AI use and submission arrangements.

In [ ]:
# Research 4: sources, example, reasoning and one limitation.


<a id="finish"></a>
## Finish and save

- [ ] I completed the three short Python practice tasks; CLI work is optional.
- [ ] My row validator passes one valid case and one conversion-failure case.
- [ ] My six-row audit uses the supplied grouping rule and accounts for every row.
- [ ] I can explain one proposed, held and rejected record without claiming availability.
- [ ] I kept Research 3 feedback and prepared Research 4 for Week 6.

**Save now:** save this notebook and your created files, then keep a backup. Email completed work to [submit.progmis@gmail.com](mailto:submit.progmis@gmail.com); do not post full assessed solutions in shared Issues. If something is unfinished, record the exact input, expectation, observation and next step.

[Research preparation](#research) · [Start here](#start) · [Session 1](#session-1) · [Session 2](#session-2) · [Project studio](#project-studio) · [Finish](#finish)